# 🚀 CTR Prediction - Google Colab Training

**Before running this notebook:**
1. Upload `criteo_processed.tar.gz` to your Google Drive
2. Upload `ctr_code.tar.gz` to your Google Drive
3. Enable GPU: Runtime → Change runtime type → GPU → T4 GPU

**Important:** Run cells in order!

## 📁 Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 🔧 Step 2: Setup Environment

In [ ]:
# Create working directory
!mkdir -p /content/CTR-Prediction
%cd /content/CTR-Prediction

# Install dependencies
!pip install -q numpy pandas pyarrow scikit-learn joblib

# Check PyTorch and GPU
import torch
print(f"✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU")

## 📦 Step 3: Extract Data

**⚠️ IMPORTANT: Update the path below to match your Google Drive structure!**

In [ ]:
# Update this path to match where you uploaded the file in Google Drive
DATA_ARCHIVE = "/content/drive/MyDrive/CTR-Prediction/criteo_processed.tar.gz"

# Extract data
print("📦 Extracting processed data...")
!tar -xzf {DATA_ARCHIVE}

# Verify extraction
print("\n📊 Verifying data files:")
!echo "Parquet files: $(ls data/processed/*.parquet | wc -l)"
!echo "Metadata files:"
!ls -lh data/processed/metadata/

print("\n✅ Data extracted successfully!")

## 💻 Step 4: Extract Code

In [ ]:
# Update this path to match where you uploaded the code archive
CODE_ARCHIVE = "/content/drive/MyDrive/CTR-Prediction/ctr_code.tar.gz"

print("📦 Extracting source code...")
!tar -xzf {CODE_ARCHIVE}

# Verify code extraction
print("\n📂 Project structure:")
!ls -la
print("\n📂 Source modules:")
!ls src/
print("\n📂 Training scripts:")
!ls scripts/

print("\n✅ Code extracted successfully!")

## 🔧 Step 4.5: Install Project as Python Package

**⚠️ CRITICAL: This step is required to fix "ModuleNotFoundError: No module named 'src'"**

In [ ]:
# Install the project in editable mode
print("📦 Installing project package...")
!pip install -e .

# Verify installation by importing modules
print("\n🔍 Testing imports...")
try:
    from src.data.module import CriteoBatchIterator
    from src.models.baselines import LogisticRegressionModel, ShallowMLP
    from src.eval.metrics import compute_auc, compute_logloss
    print("✅ All modules imported successfully!")
    print("✅ Ready to train models!")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("Please check that setup.py exists and try again.")

## 🏋️ Step 5: Train Models

### 5.1 Logistic Regression (No Embeddings - Baseline)

In [ ]:
!python scripts/train_baselines.py \
    --model lr \
    --no-embeddings \
    --epochs 5 \
    --device cuda \
    --batch-size 4096 \
    --log-file reports/lr_no_emb.csv

### 5.2 Logistic Regression (With Embeddings, dim=8)

In [ ]:
!python scripts/train_baselines.py \
    --model lr \
    --embedding-dim 8 \
    --epochs 5 \
    --device cuda \
    --batch-size 4096 \
    --log-file reports/lr_emb8.csv

### 5.3 MLP (With Embeddings, dim=8)

In [ ]:
!python scripts/train_baselines.py \
    --model mlp \
    --embedding-dim 8 \
    --epochs 5 \
    --device cuda \
    --batch-size 4096 \
    --dropout 0.2 \
    --log-file reports/mlp_emb8.csv

### 5.4 MLP (With Embeddings, dim=16)

In [ ]:
!python scripts/train_baselines.py \
    --model mlp \
    --embedding-dim 16 \
    --epochs 5 \
    --device cuda \
    --batch-size 4096 \
    --dropout 0.2 \
    --log-file reports/mlp_emb16.csv

### 5.5 DeepFM (Optional - Requires more time)

In [ ]:

!python scripts/train_deepfm.py \
    --epochs 5 \
    --device cuda \
    --batch-size 4096 \
    --embedding-dim 16 \
    --mixed-precision


## 📊 Step 6: View Results

In [ ]:
import pandas as pd

# Load all results
results = {}

for model_name in ["lr_no_emb", "lr_emb8", "mlp_emb8", "mlp_emb16"]:
    csv_path = f"reports/{model_name}.csv"
    try:
        df = pd.read_csv(csv_path)
        results[model_name] = df
        print(f"\n{'='*60}")
        print(f"📈 {model_name.upper()}")
        print(f"{'='*60}")
        print(df.to_string(index=False))
        print(f"\nBest Val AUC: {df['val_auc'].max():.6f}")
    except FileNotFoundError:
        print(f"⚠️  {model_name}.csv not found")

## 📊 Step 7: Compare Models (Visualization)

In [ ]:
import matplotlib.pyplot as plt

if results:
    # Plot AUC curves
    plt.figure(figsize=(12, 5))

    # Plot 1: Val AUC
    plt.subplot(1, 2, 1)
    for name, df in results.items():
        plt.plot(df['epoch'], df['val_auc'], marker='o', label=name, linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Validation AUC', fontsize=12)
    plt.title('Validation AUC Comparison', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot 2: Val Loss
    plt.subplot(1, 2, 2)
    for name, df in results.items():
        plt.plot(df['epoch'], df['val_loss'], marker='o', label=name, linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Validation Loss', fontsize=12)
    plt.title('Validation Loss Comparison', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('reports/model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n✅ Plot saved to reports/model_comparison.png")
else:
    print("⚠️  No results to plot. Please train models first.")

## 💾 Step 8: Save Results to Google Drive

In [ ]:
# Create results folder in Drive
!mkdir -p /content/drive/MyDrive/CTR-Prediction/results

# Copy all results
!cp reports/*.csv /content/drive/MyDrive/CTR-Prediction/results/ 2>/dev/null || echo "No CSV files found"
!cp reports/*.png /content/drive/MyDrive/CTR-Prediction/results/ 2>/dev/null || echo "No PNG files found"
!cp reports/*.pt /content/drive/MyDrive/CTR-Prediction/results/ 2>/dev/null || echo "No model checkpoints found"

print("\n✅ Results saved to Google Drive!")
print("\n📁 Saved files:")
!ls -lh /content/drive/MyDrive/CTR-Prediction/results/

## 📝 Step 9: Generate Summary Report

In [ ]:
if results:
    # Generate summary table
    summary = []
    for name, df in results.items():
        last_epoch = df.iloc[-1]
        summary.append({
            'Model': name,
            'Best Val AUC': df['val_auc'].max(),
            'Final Val AUC': last_epoch['val_auc'],
            'Final Val Loss': last_epoch['val_loss'],
            'Final Val Logloss': last_epoch['val_logloss'],
            'Epochs': len(df)
        })

    summary_df = pd.DataFrame(summary)
    summary_df = summary_df.sort_values('Best Val AUC', ascending=False)

    print("\n" + "="*80)
    print("📊 FINAL SUMMARY")
    print("="*80)
    print(summary_df.to_string(index=False))
    print("="*80)

    # Save summary
    summary_df.to_csv('reports/summary.csv', index=False)
    !cp reports/summary.csv /content/drive/MyDrive/CTR-Prediction/results/
    print("\n✅ Summary saved to Google Drive!")
else:
    print("⚠️  No results to summarize. Please train models first.")

## 🎉 All Done!

Your trained models and results are now saved in:
- **Colab**: `/content/CTR-Prediction/reports/`
- **Google Drive**: `MyDrive/CTR-Prediction/results/`

### Next Steps:
1. Download the results from Google Drive
2. Analyze the model comparisons
3. Update your project report with the findings
4. Consider training DeepFM for even better performance!